In [3]:
import re
import string
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


data = {
    'text': [
        "Breaking: Alien spacecraft lands in New York City today!",
        "The Federal Reserve announced an increase in interest rates by 0.25 percent.",
        "Miracle herb cures all known diseases instantly without side effects.",
        "The local council approved funding for a new public library project.",
        "Secret government plot exposed: Earth is actually flat!",
        "NASA launches a new telescope to study distant galaxies."
    ],
    'label': [1, 0, 1, 0, 1, 0]  
}

df = pd.DataFrame(data)


def clean_text(text):
    text = text.lower()                                    
    text = re.sub(r'https?://\S+|www\.\S+', '', text)     
    text = re.sub(r'<.*?>+', '', text)                    
    text = re.sub(f'[{string.punctuation}]', '', text)    
    text = re.sub(r'\n', '', text)                        
    text = re.sub(r'\w*\d\w*', '', text)                  
    return text

df['clean_text'] = df['text'].apply(clean_text)


X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


model = LogisticRegression()
model.fit(X_train_vec, y_train)


y_pred = model.predict(X_test_vec)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))


def predict_news(news_headline):
    cleaned = clean_text(news_headline)
    vec = vectorizer.transform([cleaned])
    prediction = model.predict(vec)[0]
    confidence = model.predict_proba(vec).max() * 100
    
    label = "FAKE News 🚨" if prediction == 1 else "REAL News ✅"
    print(f"Headline: '{news_headline}'")
    print(f"Prediction: {label} ({confidence:.1f}% confidence)\n")


predict_news("Scientists discover drinking salt water turns humans invisible!")
predict_news("The stock market saw slight gains following the quarterly economic report.")

Accuracy: 0.00%

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       1.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

Headline: 'Scientists discover drinking salt water turns humans invisible!'
Prediction: FAKE News 🚨 (50.4% confidence)

Headline: 'The stock market saw slight gains following the quarterly economic report.'
Prediction: FAKE News 🚨 (50.4% confidence)



In [5]:
import joblib
joblib.dump(model,"housing.pkl")
print("Model Saved Successfully")

Model Saved Successfully


In [7]:
!pip install streamlit

Defaulting to user installation because normal site-packages is not writeable
